# 02 — Train YOLOP Vehicle + Lane Baseline

**Purpose:** Full training loop for the YOLOP-style VehicleLane model on BDD100K.
Checkpoints and metrics are saved to Drive for cross-session persistence.

**Architecture:** Shared CSP encoder + FPN → Detection PAN (nc=5) + Lane seg decoder (2-ch)

**Training:**
- Adam optimizer, cosine LR with warmup
- AMP (mixed precision)
- 100 epochs default, validate every epoch

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!pip install -q yacs tqdm opencv-python-headless tensorboard

In [ ]:
import os, sys
REPO_ROOT = '/content/drive/MyDrive/EcoCAR/EcoCAR-Perception-Pipeline-YOLO26-BDD100K/yolop_vehicle_lane'
os.chdir(REPO_ROOT)
sys.path.insert(0, REPO_ROOT)

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import logging
import math
from pathlib import Path
from torch.cuda import amp
from torch.utils.data import DataLoader
from torch.utils.tensorboard import SummaryWriter
import torchvision.transforms as T

from lib.config import cfg
from lib.models import get_net
from lib.core import get_loss, train, validate
from lib.dataset import BddDataset

print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

In [ ]:
# ── Configuration ──
cfg.defrost()

# Dataset paths
# Preferred packaged root (DETR_GeoLane-style).
cfg.DATASET.ROOT = '/content/bdd100k_vehicle5'
# Raw BDD fallbacks still work if ROOT does not exist.
cfg.DATASET.DATAROOT = '/content/bdd100k/images/100k'
cfg.DATASET.LABELROOT = '/content/bdd100k/labels/100k'
cfg.DATASET.LANEROOT = '/content/bdd100k/lane_masks'

# Drive persistence
cfg.DRIVE.ROOT = '/content/drive/MyDrive/EcoCAR'
cfg.DRIVE.CHECKPOINT_DIR = '/content/drive/MyDrive/EcoCAR/checkpoints'
cfg.DRIVE.METRICS_DIR = '/content/drive/MyDrive/EcoCAR/metrics'

# Training params
cfg.TRAIN.BATCH_SIZE_PER_GPU = 16
cfg.TRAIN.END_EPOCH = 100
cfg.TRAIN.LR0 = 0.001
cfg.WORKERS = 4
cfg.LOSS.MULTI_HEAD_LAMBDA = [1.0, 1.0, 1.0, 1.0, 1.0]

cfg.freeze()

# Create output dirs
os.makedirs(cfg.DRIVE.CHECKPOINT_DIR, exist_ok=True)
os.makedirs(cfg.DRIVE.METRICS_DIR, exist_ok=True)
output_dir = cfg.DRIVE.METRICS_DIR
tb_log_dir = os.path.join(cfg.DRIVE.ROOT, 'tb_logs')
os.makedirs(tb_log_dir, exist_ok=True)

In [ ]:
# ── Logger ──
logger = logging.getLogger('train')
logger.setLevel(logging.INFO)
if not logger.handlers:
    ch = logging.StreamHandler()
    ch.setLevel(logging.INFO)
    logger.addHandler(ch)

In [ ]:
# ── Build datasets ──
transform = T.ToTensor()

train_dataset = BddDataset(cfg, is_train=True, inputsize=640, transform=transform)
val_dataset = BddDataset(cfg, is_train=False, inputsize=640, transform=transform)

train_loader = DataLoader(
    train_dataset, batch_size=cfg.TRAIN.BATCH_SIZE_PER_GPU,
    shuffle=cfg.TRAIN.SHUFFLE, num_workers=cfg.WORKERS,
    pin_memory=cfg.PIN_MEMORY, collate_fn=train_dataset.collate_fn
)
val_loader = DataLoader(
    val_dataset, batch_size=cfg.TEST.BATCH_SIZE_PER_GPU,
    shuffle=False, num_workers=cfg.WORKERS,
    pin_memory=cfg.PIN_MEMORY, collate_fn=val_dataset.collate_fn
)

print(f'Train: {len(train_dataset)} samples, {len(train_loader)} batches')
print(f'Val:   {len(val_dataset)} samples, {len(val_loader)} batches')

In [ ]:
# ── Build model, loss, optimizer ──
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

model = get_net(cfg).to(device)
model.gr = 1.0  # iou loss ratio
model.nc = 5
model.names = cfg.MODEL.VEHICLE_CLASSES

criterion = get_loss(cfg, device)

# Optimizer
if cfg.TRAIN.OPTIMIZER == 'adam':
    optimizer = torch.optim.Adam(model.parameters(), lr=cfg.TRAIN.LR0,
                                betas=(cfg.TRAIN.MOMENTUM, 0.999))
else:
    optimizer = torch.optim.SGD(model.parameters(), lr=cfg.TRAIN.LR0,
                               momentum=cfg.TRAIN.MOMENTUM,
                               weight_decay=cfg.TRAIN.WD,
                               nesterov=cfg.TRAIN.NESTEROV)

# Set initial_lr for warmup
for pg in optimizer.param_groups:
    pg['initial_lr'] = cfg.TRAIN.LR0

# LR scheduler (cosine)
lf = lambda x: ((1 + math.cos(x * math.pi / cfg.TRAIN.END_EPOCH)) / 2) * \
               (1 - cfg.TRAIN.LRF) + cfg.TRAIN.LRF
scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda=lf)

# AMP scaler
scaler = amp.GradScaler(enabled=device.type != 'cpu')

num_params = sum(p.numel() for p in model.parameters())
print(f'Model params: {num_params/1e6:.2f}M')
print(f'Device: {device}')

In [ ]:
# ── Resume from checkpoint if available ──
start_epoch = cfg.TRAIN.BEGIN_EPOCH
best_map = 0.0
best_ll_iou = 0.0

ckpt_path = os.path.join(cfg.DRIVE.CHECKPOINT_DIR, 'latest.pth')
if os.path.exists(ckpt_path) and cfg.AUTO_RESUME:
    ckpt = torch.load(ckpt_path, map_location=device)
    model.load_state_dict(ckpt['state_dict'])
    optimizer.load_state_dict(ckpt['optimizer'])
    start_epoch = ckpt['epoch'] + 1
    best_map = ckpt.get('best_map', 0.0)
    best_ll_iou = ckpt.get('best_ll_iou', 0.0)
    print(f'Resumed from epoch {start_epoch}, best_map={best_map:.4f}, best_ll_iou={best_ll_iou:.4f}')
else:
    print('Training from scratch')

In [ ]:
# ── Training loop ──
writer = SummaryWriter(tb_log_dir)
writer_dict = {
    'writer': writer,
    'train_global_steps': start_epoch * len(train_loader),
}

num_batch = len(train_loader)
num_warmup = max(round(cfg.TRAIN.WARMUP_EPOCHS * num_batch), 1000)

for epoch in range(start_epoch, cfg.TRAIN.END_EPOCH):
    # Train
    train(cfg, train_loader, model, criterion, optimizer, scaler,
          epoch, num_batch, num_warmup, writer_dict, logger, device)

    scheduler.step()

    # Validate
    if epoch % cfg.TRAIN.VAL_FREQ == 0:
        ll_seg_result, det_result, val_loss, maps, times = validate(
            epoch, cfg, val_loader, val_dataset, model, criterion,
            output_dir, tb_log_dir, writer_dict, logger, device
        )

        ll_acc, ll_iou, ll_miou = ll_seg_result
        mp, mr, map50, map_all = det_result

        # Log to tensorboard
        writer.add_scalar('val/loss', val_loss, epoch)
        writer.add_scalar('val/mAP50', map50, epoch)
        writer.add_scalar('val/mAP', map_all, epoch)
        writer.add_scalar('val/ll_iou', ll_iou, epoch)
        writer.add_scalar('val/ll_acc', ll_acc, epoch)

        logger.info(f'Epoch {epoch} | mAP50={map50:.4f} mAP={map_all:.4f} | '
                    f'LL_IoU={ll_iou:.4f} LL_Acc={ll_acc:.4f} | Loss={val_loss:.4f}')

        # Save checkpoint
        is_best = map50 > best_map or ll_iou > best_ll_iou
        if map50 > best_map:
            best_map = map50
        if ll_iou > best_ll_iou:
            best_ll_iou = ll_iou

        state = {
            'epoch': epoch,
            'state_dict': model.state_dict(),
            'optimizer': optimizer.state_dict(),
            'best_map': best_map,
            'best_ll_iou': best_ll_iou,
        }
        torch.save(state, os.path.join(cfg.DRIVE.CHECKPOINT_DIR, 'latest.pth'))
        if is_best:
            torch.save(state, os.path.join(cfg.DRIVE.CHECKPOINT_DIR, 'best.pth'))
            logger.info(f'  *** New best: mAP50={best_map:.4f}, LL_IoU={best_ll_iou:.4f}')

writer.close()
print(f'\nTraining complete. Best mAP50={best_map:.4f}, Best LL_IoU={best_ll_iou:.4f}')